# 🌿 BioEcoInt — Pipeline Completo de Treinamento, SOTA Benchmark & Comitê Temporal
> **Laboratório IFMaker / BioEcoInt** — *Engenharia de Software & Machine Learning Aplicada à Fauna Amazônica*
> 
> Este notebook executa a **Bateria Completa de Benchmarks Multi-Etapas** para classificação da fauna amazônica:
> 1. **Fase 1 (Benchmark de Arquiteturas Mobile)**: ResNet18 Baseline, Two-Stage YOLO Crop, ResNet18 Coordinate Attention, MobileNetV4-Conv-Small e **RepViT-M0.9 (Campeão SOTA com 80.26% Macro F1 e 86.29% Balanced Acc)**.
> 2. **Fase 2 (Benchmark de Funções de Perda)**: Cross-Entropy com Label Smoothing, Focal Loss (gamma = 2.0) e Balanced Softmax / Logit Adjustment (tau = 1.0).
> 3. **Fase 3 & 4 (Paradigma P3 - SAM 2 Tap-to-Crop)**: RepViT-M0.9 treinado sobre a Região de Interesse (ROI) recortada de alta precisão via Segment Anything 2.
> 4. **Fase 5 (Avaliação em Vídeo & Comitê Temporal)**: Simulação do uso real em vídeo de 2 segundos a 3 FPS com regra de consenso de >= 70% no modelo RepViT.
> 5. **Fase 6 (Otimização de Hiperparâmetros HPO)**: Optuna TPE com poda assíncrona ASHA focado no RepViT.
> 6. **Fase 7 (Deploy Mobile & Quantização)**: Exportação ONNX, TFLite Float16 e TFLite INT8 (PTQ com dataset representativo) do RepViT.
> 
> ---
> ⚡ **Gerenciamento de Compute Units (CUs)**: Este pipeline utiliza salvamento contínuo de checkpoints e encerramento automatizado de runtime para evitar desperdício de créditos.


### 1. Verificação de Aceleração por Hardware (GPU)
Certifique-se de que o ambiente de execução está configurado com GPU (T4, L4, V100 ou A100).


In [ ]:
import torch
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Dispositivo: {torch.cuda.get_device_name(0)}")
    print(f"Capacidade de Memória: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
!nvidia-smi


### 2. Instalação do Gerenciador de Pacotes Ultrarrápido (`uv`)
Utilizamos o `uv` da Astral para provisionar o ambiente virtual e dependências em menos de 40 segundos, garantindo isolamento total.


In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] = f"/root/.local/bin:{os.environ['PATH']}"
!uv --version


### 3. Clonagem do Repositório (`bio-interaction-ml`)
Configure seu Personal Access Token do GitHub nos Secrets do Colab (`GITHUB_TOKEN`) ou utilize acesso direto caso o repositório seja público.


In [ ]:
import os
from google.colab import userdata

try:
    github_token = userdata.get("GITHUB_TOKEN")
    repo_url = f"https://{github_token}@github.com/Emmerson2018/bio-interaction-ml.git"
    print("✓ Autenticação via GitHub Token configurada com sucesso.")
except Exception:
    repo_url = "https://github.com/Emmerson2018/bio-interaction-ml.git"
    print("ℹ GITHUB_TOKEN não encontrado nos Secrets do Colab. Tentando clonagem pública...")

if not os.path.exists("bio-interaction-ml"):
    print("Clonando repositório...")
    !git clone {repo_url} bio-interaction-ml
else:
    print("Repositório já clonado. Atualizando...")
    !cd bio-interaction-ml && git pull

%cd bio-interaction-ml
!git status


### 4. Sincronização do Ambiente Virtual (`uv sync`)
Instala PyTorch com aceleração CUDA, Torchvision, Ultralytics, Optuna, ONNX, DVC e demais dependências da toolbox.


In [ ]:
!uv sync
!uv run python -c "import torch, torchvision, ultralytics; from ultralytics import SAM; SAM('sam2_t.pt'); print('✓ Ambiente e modelo SAM 2 verificados com sucesso! PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())"


### 5. Montagem do Google Drive & Sincronização do Dataset Sanado via DVC
Montamos o Google Drive para obter as credenciais e sincronizar o dataset sanitizado (80 vídeos, 15 classes, 19.413 imagens geradas com zero vazamento entre vídeos).
O dataset é descompactado diretamente no SSD NVMe local (`/content/bio-interaction-ml/datasets/generated/`) para máxima taxa de leitura por segundo (eliminando gargalos de I/O do FUSE).


In [ ]:
from google.colab import drive
import os
import shutil
import subprocess

# 1. Montar o Google Drive
drive.mount("/content/drive")

# 2. Configurar credenciais headless para o PyDrive2 / DVC
creds_dir = "/root/.cache/pydrive2fs/580761162230-jc0qbfr9rpglof21kknm0tr8urugbh6u.apps.googleusercontent.com"
os.makedirs(creds_dir, exist_ok=True)
target_file = os.path.join(creds_dir, "default.json")

# Caminhos de busca automática no Google Drive montado
candidates = [
    "/content/drive/MyDrive/default.json",
    "/content/drive/MyDrive/BioEcoInt_DVC_Auth/default.json",
    "/content/drive/MyDrive/dvc_credentials.json",
]

found = False
for cand in candidates:
    if os.path.exists(cand):
        shutil.copy(cand, target_file)
        print(f"✓ Credenciais headless do DVC carregadas de: {cand}")
        found = True
        break

if not found:
    try:
        from google.colab import userdata
        secret_json = userdata.get("DVC_DEFAULT_JSON")
        if secret_json:
            with open(target_file, "w") as f:
                f.write(secret_json)
            print("✓ Credenciais do DVC carregadas com sucesso via Colab Secrets (DVC_DEFAULT_JSON)!")
            found = True
    except Exception:
        pass

if not found:
    print("⚠️ Arquivo de credenciais default.json não encontrado no Drive montado.")
    print("Faça o upload do arquivo default.json abaixo para autenticar o DVC sem abrir localhost:")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if fname.endswith(".json"):
            shutil.move(fname, target_file)
            print(f"✓ Credenciais instaladas com sucesso em {target_file}!")
            found = True
            break

# 3. Configurar segredo local do OAuth Client do laboratório no DVC
try:
    from google.colab import userdata
    cs = userdata.get("GDRIVE_CLIENT_SECRET")
    if cs:
        subprocess.run(["uv", "run", "dvc", "remote", "modify", "--local", "gdrive-storage", "gdrive_client_secret", str(cs)], check=False)
except Exception:
    pass

# 4. Puxar dataset sanitizado do DVC
subprocess.run(["uv", "run", "dvc", "pull", "datasets/generated/animals_video_sanitized.zip.dvc"], check=False)
if os.path.exists("datasets/generated/animals_video_sanitized.zip"):
    print("Descompactando dataset sanitizado no disco local...")
    subprocess.run(["unzip", "-q", "-o", "datasets/generated/animals_video_sanitized.zip", "-d", "datasets/generated/"], check=False)
    print("✓ Dataset sanitizado pronto!")

# 5. Puxar dataset SAM 2 crops do DVC
if os.path.exists("datasets/generated/animals_video_sam2_crops.zip.dvc"):
    print("Baixando dataset SAM 2 crops do Google Drive...")
    subprocess.run(["uv", "run", "dvc", "pull", "datasets/generated/animals_video_sam2_crops.zip.dvc"], check=False)
    if os.path.exists("datasets/generated/animals_video_sam2_crops.zip"):
        subprocess.run(["unzip", "-q", "-o", "datasets/generated/animals_video_sam2_crops.zip", "-d", "datasets/generated/"], check=False)
        print("✓ Dataset SAM 2 crops pronto para treinamento!")


### 6. 🚀 Experimento 1: ResNet18 Baseline Sanado
- **Arquitetura**: ResNet18 pré-treinada com Two-Stage Transfer Learning.
- **Divisão**: Vídeos estritamente disjuntos por classe (Zero Shuffle Leakage).
- **Otimizador**: AdamW com taxas de aprendizado diferenciais ($1.5 \times 10^{-5}$ nas camadas rasas, $4.5 \times 10^{-5}$ nas profundas e $4 \times 10^{-4}$ no classificador).
- **Augmentation**: Mixup & CutMix calibrados.


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp1_resnet18_baseline.yml


### 7. 🎯 Experimento 2 (SOTA 1): Two-Stage YOLO Crop Classifier
- **Arquitetura**: Detector YOLOv8n (estágio 1) localiza a ROI da fauna com margem dinâmica (+15% padding) $\to$ ResNet18 (estágio 2) classifica exclusivamente o animal.
- **Benefício**: Elimina ruído de fundo amazônico denso, troncos e folhagens que confundem alvos camuflados.


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp2_resnet18_yolo_crop.yml


### 8. 🧠 Experimento 3 (SOTA 2): ResNet18 com Coordinate Attention (CA)
- **Arquitetura**: ResNet18 re-arquitetada integrando blocos de Atenção Coordenada 2D (Hou et al., CVPR) após as convoluções dos estágios Layer3 e Layer4.
- **Benefício**: Codifica simultaneamente relações espaciais horizontais e verticais de longo alcance sem o custo quadrático de Vision Transformers, preservando pequenos animais em resolução mobile.


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp3_resnet18_ca.yml


### 9. ⚡ Experimento 4 (SOTA 3): Google MobileNetV4-Conv-Small com Balanced Softmax & EMA
- **Arquitetura**:  (3.8M parâmetros) projetada pelo Google para inferência mobile ultrarrápida (<5 ms).
- **Função de Perda**: **Balanced Softmax / Logit Adjustment** com $\tau = 1.0$, mitigando o desbalanceamento das espécies raras (, ).
- **Estabilidade Numérica**: **Model EMA** ($\beta = 0.999$), Differential Learning Rate e Weight Decay Seletivo (0.0 para bias/norm).


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_mobilenetv4_small.yml


### 10. 🚀 Experimento 5 (SOTA 4): RepViT-M0.9 com Balanced Softmax & EMA
- **Arquitetura**:  (4.7M parâmetros) com reparametrização estrutural no deploy mobile.
- **Otimização**: Balanced Softmax ($\tau = 1.0$), Model EMA e Two-Stage Warmup.


In [ ]:
!MPLBACKEND=Agg uv run base-train -opt options/exp_repvit_m0_9.yml


### 11. 🔬 Experimento 6 (Fase 2): Benchmark de Funções de Perda (Iso-condicional na MobileNetV4)
Compara o impacto direto da função objetivo sob a mesma arquitetura campeã (**MobileNetV4-Conv-Small**) com EMA e Weight Decay Seletivo:
1. **Cross-Entropy Clássica** com Label Smoothing (0.1);
2. **Focal Loss** ($\gamma = 2.0$) para forçar o foco nos gradientes de amostras difíceis/camufladas;
3. *(O experimento 4 acima serviu como a condição com Balanced Softmax / Logit Adjustment $\tau=1.0$)*.


In [ ]:
# 1. Treino com Cross-Entropy pura + Label Smoothing 0.1
!MPLBACKEND=Agg uv run base-train -opt options/exp_loss_ce.yml

# 2. Treino com Focal Loss (gamma=2.0) + Label Smoothing 0.1
!MPLBACKEND=Agg uv run base-train -opt options/exp_loss_focal.yml


### 12. 🎯 Experimento 7 (Fase 4): Paradigma P3 — RepViT-M0.9 Campeão Treinado sobre ROI Recortada via SAM 2
- **Conceito**: No lugar de treinar sobre a imagem inteira com ruído de fundo amazônico, treinamos a arquitetura campeã (**RepViT-M0.9**) diretamente sobre os recortes da fauna isolados pelo SAM 2 (+15% de margem dinâmica).
- **Vantagem**: Une o campo receptivo global e o viés indutivo do Vision Transformer à eliminação cirúrgica do ruído de fundo, maximizando o reconhecimento de espécies difíceis e camufladas.


In [ ]:
# Treino do RepViT-M0.9 (Campeão) sobre os recortes SAM 2
!MPLBACKEND=Agg uv run base-train -opt options/exp_repvit_m0_9_sam2_crops.yml

# (Opcional) Treino da MobileNetV4 sobre recortes SAM 2 para comparação
# !MPLBACKEND=Agg uv run base-train -opt options/exp_mobilenetv4_sam2_crops.yml


### 13. 📹 Experimento 8 (Fase 5): Benchmark de Avaliação em Vídeo & Comitê Temporal (2s @ 3 FPS)
Simula o comportamento exato do usuário no aplicativo Flutter em campo:
- O usuário aponta a câmera por **2 segundos** (6 quadros a 3 FPS);
- O modelo classifica cada quadro individualmente;
- O **Comitê Temporal** agrega os votos: se a classe mais votada tiver **$\ge 70\%$ dos votos** (ex.: 5 em 6 ou 6 em 6), a predição é aceita;
- Se não houver consenso, o sistema se abstém ou solicita novo enquadramento, eliminando falsos positivos na borda.


In [ ]:
# 1. Comitê Temporal no Campeão RepViT Direto (Dataset Sanitized)
!uv run base-test-committee -opt options/exp_repvit_m0_9.yml -c experiments/Exp_RepViT_M0_9_BalancedSoftmax/models/best_macro_f1.pth --manifest datasets/generated/animals_video_sanitized/dataset_manifest.json --threshold 0.70 --fps 3.0 --duration 2.0 --output experiments/Exp_RepViT_M0_9_BalancedSoftmax/committee_direct_report.json

# 2. Comitê Temporal no Campeão RepViT treinado sobre recortes SAM 2 (com Exemplar Matching Anti-Oclusão)
!uv run base-test-committee -opt options/exp_repvit_m0_9_sam2_crops.yml -c experiments/Exp_RepViT_M0_9_SAM2_Crops/models/best_macro_f1.pth --manifest datasets/generated/animals_video_sam2_crops/dataset_manifest.json --use-sam2 --threshold 0.70 --fps 3.0 --duration 2.0 --sim-thresh 0.50 --output experiments/Exp_RepViT_M0_9_SAM2_Crops/committee_sam2_report.json


### 14. ⚙️ Experimento 9 (Fase 6): Otimização de Hiperparâmetros HPO (Optuna + ASHA Pruning)
Executa busca bayesiana TPE com poda agressiva por Sucessivas Reduções (ASHA) para calibrar `head_lr`, taxa diferencial do backbone, `weight_decay` e regularização de dropout no modelo campeão.


In [ ]:
# HPO Optuna focado na arquitetura campeã RepViT-M0.9
!uv run python experiments/hpo_optuna_baseline.py -opt options/exp_repvit_m0_9.yml --n-trials 15 --epochs-per-trial 5


### 15. 📱 Exportação Mobile e Quantização (TFLite FP16 & INT8 PTQ)
Converte os melhores modelos treinados para formatos otimizados de borda (ONNX e TFLite), medindo tamanho e latência de inferência sob as restrições mobile do projeto.


In [ ]:
# Exportando os modelos campeões RepViT e baselines para TFLite FP16 e INT8 PTQ
!MPLBACKEND=Agg uv run base-export -opt options/exp_repvit_m0_9.yml
!MPLBACKEND=Agg uv run base-export -opt options/exp_repvit_m0_9_sam2_crops.yml
!MPLBACKEND=Agg uv run base-export -opt options/exp_mobilenetv4_small.yml
!MPLBACKEND=Agg uv run base-export -opt options/exp1_resnet18_baseline.yml


### 16. 📊 Comparativo Consolidado de Métricas dos 3 Experimentos
Gera tabela e gráficos comparando acurácia balanceada, Macro F1, tamanho em disco e taxa de acerto por classe.


In [ ]:
import os
import glob
import json
import pandas as pd
import matplotlib.pyplot as plt

exp_dirs = [
    "experiments/Exp1_ResNet18_Baseline_Sanitized",
    "experiments/Exp2_ResNet18_YOLO_Crop",
    "experiments/Exp3_ResNet18_CoordinateAttention",
    "experiments/Exp_MobileNetV4_Small_BalancedSoftmax",
    "experiments/Exp_RepViT_M0_9_BalancedSoftmax",
    "experiments/Exp_Loss_CE_MobileNetV4",
    "experiments/Exp_Loss_Focal_MobileNetV4",
    "experiments/Exp_MobileNetV4_SAM2_Crops",
    "experiments/Exp_RepViT_M0_9_SAM2_Crops",
]

results = []
for d in exp_dirs:
    metrics_file = os.path.join(d, "val_metrics.json")
    test_summary = os.path.join(d, "test_results", "test_summary.json")
    log_files = glob.glob(os.path.join(d, "*.log"))
    exp_name = os.path.basename(d)
    if os.path.exists(test_summary):
        with open(test_summary) as f:
            tdata = json.load(f)
            results.append({
                "Experimento": exp_name,
                "Acurácia Teste": round(tdata.get("test_accuracy", 0) * 100, 2),
                "Macro F1": round(tdata.get("metrics_per_class", {}).get("macro avg", {}).get("f1-score", 0) * 100, 2),
                "Fonte": "test_summary.json"
            })
    elif os.path.exists(metrics_file):
        with open(metrics_file) as f:
            data = json.load(f)
            results.append({"Experimento": exp_name, **data, "Fonte": "val_metrics.json"})
    elif log_files:
        results.append({"Experimento": exp_name, "Status": "Concluído (Ver log)"})

if results:
    df = pd.DataFrame(results)
    display(df)
else:
    print("Nenhum arquivo de métricas encontrado ainda.")

# Exibir relatórios do Comitê Temporal se disponíveis
committee_paths = [
    "experiments/Exp_RepViT_M0_9_BalancedSoftmax/committee_direct_report.json",
    "experiments/Exp_RepViT_M0_9_SAM2_Crops/committee_sam2_report.json",
    "experiments/Exp_MobileNetV4_Small_BalancedSoftmax/committee_direct_report.json",
    "experiments/Exp_MobileNetV4_SAM2_Crops/committee_sam2_report.json",
    "experiments/committee_mobilenetv4_direct.json",
    "experiments/committee_mobilenetv4_sam2.json",
]
for rep_path in committee_paths:
    if os.path.exists(rep_path):
        with open(rep_path) as f:
            rep_data = json.load(f)
        print()
        print(f"=== Relatório do Comitê: {rep_path} ===")
        print("Single-Frame Acc:", rep_data.get("single_frame_metrics", {}).get("accuracy"))
        print("Committee Acc (decided):", rep_data.get("committee_metrics", {}).get("accuracy_on_decided"))
        print("Abstention Rate:", rep_data.get("committee_metrics", {}).get("abstention_rate"))


### 17. 💾 Sincronização de Resultados e Checkpoints
Salva os pesos pesados no Google Drive via DVC e comita as métricas leves (relatórios e matrizes de confusão) no repositório Git.


In [ ]:
# 1. Versionar modelos treinados e exports no DVC usando Python seguro (sem falha de interpolação do shell)
import glob
import subprocess

for p in sorted(glob.glob('experiments/*/models') + glob.glob('experiments/*/export')):
    print(f'Adicionando ao DVC: {p}')
    subprocess.run(['uv', 'run', 'dvc', 'add', p], check=False)

print('Enviando artefatos pesados para o Google Drive via DVC...')
subprocess.run(['uv', 'run', 'dvc', 'push'], check=False)

# 2. Comitar relatórios leves no Git de forma segura
from google.colab import userdata
try:
    github_token = userdata.get('GITHUB_TOKEN')
    !git config --global user.name "BioEcoInt Lab Worker"
    !git config --global user.email "sergioaugusto.bezerra.jr@gmail.com"
    !git remote set-url origin https://{github_token}@github.com/Emmerson2018/bio-interaction-ml.git
    !git add experiments/*/*.dvc experiments/*/test_results/ experiments/*/visualization/ experiments/*/committee_*.json
    !git commit -m "feat(colab): treinamento e modelos DVC concluidos" || true
    !git push origin main || echo "⚠️ Aviso: Push para GitHub rejeitado. Os modelos estão salvos no Google Drive via DVC!"
except Exception as e:
    print(f"ℹ️ Git push não configurado: {e}")


### 18. 🛑 Desconexão Automática do Runtime (Proteção de CUs)
Para evitar que o notebook continue consumindo Compute Units (CUs) do seu plano Google AI Pro após o término dos treinamentos, a célula abaixo encerra a sessão da VM automaticamente.


In [ ]:
from google.colab import runtime

AUTO_DISCONNECT = True  # Altere para False se quiser inspecionar os arquivos antes de fechar

if AUTO_DISCONNECT:
    print("🛑 Todos os treinamentos foram concluídos! Desconectando o runtime para economizar CUs...")
    runtime.unassign()
else:
    print("⚠️ Desconexão automática desabilitada. O runtime continua ativo consumindo CUs.")
